# Лабораторная работа №9: Речь и текстовые цепочки

[Исходное задание](https://colab.research.google.com/drive/1S3GKYyd1c9wUKOFmfOdHmNgEVR3gMT4H)

Выполнение через совместимый API RouterAI. API-ключ берётся из окружения или локального `.env`. Сохранённые результаты получены реальными запросами.



## Задание

##Задание на семинар
1. Есть аудиофайл с записью речи. Модель распознает речь и преобразует ее в текст. Затем кодом программы проверяем длину полученного текста: если количество символов превышает 500, используется LLM с цепочкой chain1 для саммаризации текста до 100 символов; если текст короче 100 символов, применяется цепочка chain2 для его расширения до 100 символов; при длине от 100 до 500 символов текст остается без изменений. Далее выполняется синтез речи из обработанного текста.
1. Есть текст и аудиофайл с записью речи. Модель распознает речь и преобразует ее в текст. LLM с цепочкой chain1 получает 2 текста и сравнивает, а затем и выдает уровень сходства (низкий, высокий). Если сходство низкое, то цепочкой chain2 преобразуем исходный текст и делаем его максимально похожим на распознанный, а затем преобразуем в речь. Если сходство высокое, то сразу преобразуем в речь.

## Подключение

In [ ]:
from pathlib import Path
import sys

# При открытии из подпапки находим общий модуль конфигурации.
for candidate in [Path.cwd(), Path.cwd().parent]:
    if (candidate / "lab_runtime.py").exists():
        sys.path.insert(0, str(candidate))
        break

from lab_runtime import make_llm, json_result, ASSETS
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown, Image, Audio

llm = make_llm()
print("Модель:", llm.model_name)


## 1. Входная речь и её распознавание

Тестовый речевой файл создан синтезом, затем распознан отдельной STT-моделью. Подстановка заранее известной расшифровки не используется.

In [ ]:
from lab_runtime import synthesize, transcribe
source_text = "Нейронные сети помогают распознавать изображения и анализировать текст."
audio_path = synthesize(source_text, "speech_input_short.mp3")
recognized = transcribe(audio_path)
print("Распознано:", recognized, "Символов:", len(recognized))
display(Audio(filename=str(audio_path)))


## 2. Обработка длины и синтез результата

In [ ]:
chain1 = PromptTemplate.from_template("Сократи текст до 100 символов или меньше, сохрани смысл, верни только текст: {text}") | llm | StrOutputParser()
length_llm = make_llm(model="openai/gpt-4.1-mini", temperature=0)
chain2 = PromptTemplate.from_template("Расширь текст до одного законченного предложения длиной 90-100 символов. Сохрани смысл. Верни только предложение, без кавычек: {text}") | length_llm | StrOutputParser()

def process_text(text):
    if len(text) > 500:
        result = chain1.invoke({"text": text}).strip()
        for _ in range(2):
            if len(result) <= 100: break
            result = chain1.invoke({"text": result}).strip()
        assert len(result) <= 100
        return result, "chain1: сокращение"
    if len(text) < 100:
        result = chain2.invoke({"text": text}).strip()
        for _ in range(4):
            if 90 <= len(result) <= 100: break
            result = length_llm.invoke(
                f"Текст: {result}\nСейчас {len(result)} символов. Переформулируй в законченное предложение длиной 90-100 символов. Только текст."
            ).content.strip()
        assert 90 <= len(result) <= 100
        return result, "chain2: расширение"
    return text, "без изменения"

processed, route = process_text(recognized)
print("Ветка:", route, "\nРезультат:", processed, "\nСимволов:", len(processed))
speech = synthesize(processed, "lab09_processed.mp3")
display(Audio(filename=str(speech)))

# Дополнительно проверяем длинную и среднюю ветки.
for sample in [(recognized + " ") * 10, " ".join([recognized] * 3)]:
    result, route = process_text(sample)
    print("Проверка:", len(sample), "→", len(result), route, result)


## 3. Сравнение исходного и распознанного текстов

In [ ]:
similarity_llm = make_llm(model="openai/gpt-4.1-mini", temperature=0)
similarity_chain = ChatPromptTemplate.from_messages([
    ("system", 'Сравни смысл двух текстов. Почти одинаковые тексты с изменением формы слова имеют ВЫСОКОЕ сходство. Разные темы имеют НИЗКОЕ сходство. Верни JSON {{"level":"высокий","reason":"кратко"}} либо {{"level":"низкий","reason":"кратко"}}.'),
    ("human", "Первый: {original}\nВторой: {recognized}")
]) | similarity_llm.bind(response_format={"type":"json_object"}) | StrOutputParser()
rewrite_chain = PromptTemplate.from_template("Замени содержание исходного текста так, чтобы оно совпадало по смыслу с образцом. Если темы разные, используй тему образца. Только результат. Исходный: {original}. Образец: {recognized}") | similarity_llm | StrOutputParser()

for index, original in enumerate([source_text, "Сегодня обсуждаем прогноз погоды и прогулку в парке."]):
    result = json_result(similarity_chain.invoke({"original": original, "recognized": recognized}))
    if result["level"].lower() == "низкий":
        output = rewrite_chain.invoke({"original": original, "recognized": recognized})
    else:
        output = original
    print("Сравнение:", result, "\nТекст для озвучки:", output)
    assert result["level"].lower() == ("высокий" if index == 0 else "низкий")
    path = synthesize(output, f"lab09_similarity_{index}.mp3")
    display(Audio(filename=str(path)))


## Вывод

Выполнены STT и TTS через API. Цепочки меняют длину текста и сравнивают смысл. Параметр voice='eve' задаёт голос синтеза; это выбор конкретного голоса, а не отдельный пол пользователя.